# createTime, Text_TR,Region,city

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-34-09-183_textready_analysis.xlsx
/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-06-43-625_textready_analysis.xlsx
/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-17-06-320_textready_cleaned.xlsx
/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-37-52-643_textready_analysis.xlsx
/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/Tik Data/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-36-16-596_textready_analysis.xlsx
/kaggle/input/dataset

In [2]:
import os, glob, re, pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix,f1_score
from sklearn.utils import resample

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D, GlobalMaxPooling1D
from tensorflow.keras.callbacks import ReduceLROnPlateau

2026-05-21 16:04:03.450984: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779379443.730717      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779379443.804347      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779379444.473418      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779379444.473459      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779379444.473461      16 computation_placer.cc:177] computation placer alr

In [3]:
# =========================================================
# (0) PATH + SETTINGS  (ALL REGIONS) + QUICK STRUCTURE CHECK
# =========================================================
ALL_REGIONS_ROOT = "/kaggle/input/datasets/ziyadaltalhi/tik-tok-data/Tik Tok Datasets - After Processing/"
TEXT_COL = "Text_TR"  # ✅ العمود المراد العمل عليه
STAR_CANDIDATES = {"stars", "Stars"}
print("\n[STEP 0] Path checks (ALL REGIONS)")
print("Root exists:", os.path.exists(ALL_REGIONS_ROOT))
print("Root is dir:", os.path.isdir(ALL_REGIONS_ROOT))

region_dirs = sorted([d for d in glob.glob(os.path.join(ALL_REGIONS_ROOT, "*")) if os.path.isdir(d)])
print("Region folders found:", len(region_dirs))
print("Regions:", [os.path.basename(d) for d in region_dirs])

# عرض سريع لأول 3 مناطق: عدد المدن داخل كل منطقة
for rd in region_dirs[:3]:
    city_dirs = sorted([d for d in glob.glob(os.path.join(rd, "*")) if os.path.isdir(d)])
    print(f"  - {os.path.basename(rd)}: cities={len(city_dirs)} (sample: {[os.path.basename(x) for x in city_dirs[:5]]})")



[STEP 0] Path checks (ALL REGIONS)
Root exists: True
Root is dir: True
Region folders found: 5
Regions: ['المنطقة الجنوبية', 'المنطقة الشرقية', 'المنطقة الشمالية', 'المنطقة الغربية', 'المنطقة الوسطى']
  - المنطقة الجنوبية: cities=4 (sample: ['بيانات منطقة الباحة - بعد المعالجة', 'بيانات منطقة جازان - بعد المعالجة', 'بيانات منطقة عسير - بعد المعالجة', 'بيانات منطقة نجران - بعد المعالجة'])
  - المنطقة الشرقية: cities=1 (sample: ['بيانات تيك توك  الشرقية  - بعد المعالجة'])
  - المنطقة الشمالية: cities=1 (sample: ['( المنطقة الشمالية ) TikTok Data -  After Preproccesing1'])


In [4]:
# =========================================================
# Helpers: extract region name (inside parentheses) + city
# =========================================================
PAREN_RE = re.compile(r"\((.*?)\)")

def extract_region_from_folder(region_folder_name: str) -> str:
    """
    يستخرج النص بين ( ) من اسم مجلد المنطقة.
    مثال: "( المنطقة الغربية ) Google Maps Data - After Cleaning" -> "المنطقة الغربية"
    إذا لم توجد أقواس يرجع اسم المجلد نفسه.
    """
    m = PAREN_RE.search(region_folder_name)
    if m:
        return m.group(1).strip()
    return region_folder_name.strip()

def list_region_folders(root_folder: str):
    region_dirs = sorted([d for d in glob.glob(os.path.join(root_folder, "*")) if os.path.isdir(d)])
    return region_dirs

def list_city_folders(region_dir: str):
    return sorted([d for d in glob.glob(os.path.join(region_dir, "*")) if os.path.isdir(d)])

def list_files_in_city(city_dir: str):
    return (
        glob.glob(os.path.join(city_dir, "*.xlsx")) +
        glob.glob(os.path.join(city_dir, "*.xls")) +
        glob.glob(os.path.join(city_dir, "*.csv"))
    )

def read_any_file(path: str) -> pd.DataFrame:
    ext = os.path.splitext(path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path)
    elif ext == ".csv":
        # ترميزات عربية شائعة
        try:
            df = pd.read_csv(path, encoding="utf-8")
        except UnicodeDecodeError:
            try:
                df = pd.read_csv(path, encoding="utf-8-sig")
            except UnicodeDecodeError:
                df = pd.read_csv(path, encoding="cp1256")
    else:
        raise ValueError(f"Unsupported extension: {ext}")
    df.columns = [str(c).strip() for c in df.columns]
    return df

def standardize_star_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    - يقبل Stars / stars / ... ويوحّدها إلى عمود اسمه 'Stars' إن وجد.
    - لا يغير القيم الآن.
    """
    colmap = {c.lower(): c for c in df.columns}
    # إذا موجود Stars بالاسم الصحيح خلاص
    if "Stars" in df.columns:
        return df

    # ابحث عن أي عمود اسمه stars case-insensitive
    if "stars" in colmap:
        df = df.rename(columns={colmap["stars"]: "Stars"})
        return df

    # مرونة إضافية: إذا فيه rating مثلا
    for cand in list(STAR_CANDIDATES):
        key = cand.lower()
        if key in colmap:
            df = df.rename(columns={colmap[key]: "Stars"})
            return df

    return df  # لم نجد عمود نجوم

In [5]:

# =========================================================
# [STEP 0] Path checks + show regions/cities counts
# =========================================================
print("\n[STEP 0] Path checks (ALL REGIONS)")
print("Root exists:", os.path.exists(ALL_REGIONS_ROOT))
print("Root is dir :", os.path.isdir(ALL_REGIONS_ROOT))

region_dirs = list_region_folders(ALL_REGIONS_ROOT)
print("Region folders found:", len(region_dirs))

# عرض أسماء المناطق بصيغة الاسم داخل الأقواس
for rd in region_dirs:
    folder_name = os.path.basename(rd)
    region_name = extract_region_from_folder(folder_name)
    print(f"- Folder: {folder_name}  --> Region: {region_name}")



[STEP 0] Path checks (ALL REGIONS)
Root exists: True
Root is dir : True
Region folders found: 5
- Folder: المنطقة الجنوبية  --> Region: المنطقة الجنوبية
- Folder: المنطقة الشرقية  --> Region: المنطقة الشرقية
- Folder: المنطقة الشمالية  --> Region: المنطقة الشمالية
- Folder: المنطقة الغربية  --> Region: المنطقة الغربية
- Folder: المنطقة الوسطى  --> Region: المنطقة الوسطى


In [6]:
import os
import pandas as pd

# =========================================================
# (0) NORMALIZE COLUMNS
# =========================================================
def normalize_columns(df):
    col_map = {
        "Date": ["Date", "publishedAtDate", "createTimeISO"]
    }

    for new_col, possible_cols in col_map.items():
        for c in possible_cols:
            if c in df.columns:
                df[new_col] = df[c]
                break

    return df


# =========================================================
# (1) GET ALL VALID FILES (RECURSIVE)
# =========================================================
def get_analysis_files_recursive(region_path):
    files = []

    for root, dirs, filenames in os.walk(region_path):
        for fname in filenames:
            name = fname.lower()
            fpath = os.path.join(root, fname)

            if (
                name.endswith((".xlsx", ".xls")) and
                "analysis" in name and
                "summary" not in name
            ):
                files.append(fpath)

    return files


# =========================================================
# (2) SCAN AVAILABILITY
# =========================================================
def scan_availability(root_folder: str):
    rows = []

    for rd in list_region_folders(root_folder):
        region_folder = os.path.basename(rd)
        region_name = extract_region_from_folder(region_folder)

        files = get_analysis_files_recursive(rd)

        rows.append({
            "Region_Folder": region_folder,
            "Region_Name": region_name,
            "Files": len(files),
            "Sample_File": files[0] if files else None
        })

    return pd.DataFrame(rows).sort_values("Files", ascending=False).reset_index(drop=True)


# =========================================================
# (3) READ ALL REGIONS (CHUNKED)
# =========================================================
def read_all_regions_chunked(root_folder: str, max_files_per_region=None, keep_columns=None):

    all_frames = []
    report_rows = []

    region_dirs = list_region_folders(root_folder)

    for r_idx, rd in enumerate(region_dirs, 1):
        region_folder = os.path.basename(rd)
        region_name = extract_region_from_folder(region_folder)

        region_files = get_analysis_files_recursive(rd)

        if max_files_per_region is not None:
            region_files = region_files[:max_files_per_region]

        print(f"\n[STEP 1] Region {r_idx}/{len(region_dirs)}: {region_name} | files={len(region_files)}")

        n_rows_region = 0
        has_text = 0
        has_stars = 0
        has_date = 0
        read_ok = 0
        read_err = 0

        for i, p in enumerate(region_files, 1):
            try:
                df = read_any_file(p)
                df = standardize_star_column(df)
                df = normalize_columns(df)

                # استخراج اسم المدينة بشكل ذكي
                city_name = os.path.basename(os.path.dirname(p))
                if city_name.startswith("Tik_"):
                    city_name = os.path.basename(os.path.dirname(os.path.dirname(p)))

                df["Region_Folder"] = region_folder
                df["Region_Name"] = region_name
                df["City_Folder"] = city_name
                df["Source_File"] = os.path.basename(p)
                df["__path__"] = p

                # تحقق من الأعمدة
                if TEXT_COL in df.columns:
                    has_text += 1
                if "Stars" in df.columns:
                    has_stars += 1
                if "Date" in df.columns:
                    has_date += 1

                # تقليل الأعمدة (اختياري)
                if keep_columns is not None:
                    keep = [c for c in keep_columns if c in df.columns]
                    meta = ["Region_Folder","Region_Name","City_Folder","Source_File","__path__"]
                    keep = list(dict.fromkeys(keep + meta))
                    df = df[keep].copy()

                n_rows_region += len(df)
                all_frames.append(df)
                read_ok += 1

                if i <= 2:
                    print(f"  sample file [{i}] {region_name}/{city_name}/{os.path.basename(p)} rows={len(df)}")

                if i % 50 == 0:
                    print(f"  progress: {i}/{len(region_files)} files")

            except Exception as e:
                read_err += 1
                print(f"❌ Error reading: {p}")

        report_rows.append({
            "Region_Name": region_name,
            "Region_Folder": region_folder,
            "Files_Read_OK": read_ok,
            "Files_Read_Err": read_err,
            "Files_with_Text": has_text,
            "Files_with_Stars": has_stars,
            "Files_with_Date": has_date,
            "Total_Rows_This_Region": n_rows_region
        })

    df_region_report = pd.DataFrame(report_rows) \
        .sort_values("Total_Rows_This_Region", ascending=False) \
        .reset_index(drop=True)

    df_all = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()

    return df_all, df_region_report


# =========================================================
# (4) RUN
# =========================================================

print("\n[STEP 1A] Availability per region:")
df_av = scan_availability(ALL_REGIONS_ROOT)
display(df_av)

KEEP_COLS = [
    TEXT_COL,
    "Stars",
    "Date"
]

df1, df_region_report = read_all_regions_chunked(
    ALL_REGIONS_ROOT,
    max_files_per_region=None,
    keep_columns=KEEP_COLS
)

print("\n[STEP 1B] Column availability per region:")
display(df_region_report)

print("\n[STEP 1 RESULT] df1 shape:", df1.shape)
print("Columns:", df1.columns.tolist())


[STEP 1A] Availability per region:


,Region_Folder,Region_Name,Files,Sample_File
0,المنطقة الجنوبية,المنطقة الجنوبية,102,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
1,المنطقة الوسطى,المنطقة الوسطى,80,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
2,المنطقة الشمالية,المنطقة الشمالية,77,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
3,المنطقة الغربية,المنطقة الغربية,26,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
4,المنطقة الشرقية,المنطقة الشرقية,11,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...



[STEP 1] Region 1/5: المنطقة الجنوبية | files=102
  sample file [1] المنطقة الجنوبية/بيانات منطقة عسير - بعد المعالجة/Video comments 30_textready_analysis.xlsx rows=15
  sample file [2] المنطقة الجنوبية/بيانات منطقة عسير - بعد المعالجة/Video comments 19_textready_analysis.xlsx rows=25
  progress: 50/102 files
  progress: 100/102 files

[STEP 1] Region 2/5: المنطقة الشرقية | files=11
  sample file [1] المنطقة الشرقية/بيانات تيك توك  الشرقية  - بعد المعالجة/4 vid 3_textready_analysis.xlsx rows=130
  sample file [2] المنطقة الشرقية/بيانات تيك توك  الشرقية  - بعد المعالجة/4 vid 2_textready_analysis.xlsx rows=33

[STEP 1] Region 3/5: المنطقة الشمالية | files=77
  sample file [1] المنطقة الشمالية/الجوف/19_textready_analysis.xlsx rows=34
  sample file [2] المنطقة الشمالية/الجوف/20_textready_analysis.xlsx rows=10
  progress: 50/77 files

[STEP 1] Region 4/5: المنطقة الغربية | files=26
  sample file [1] المنطقة الغربية/جدة/سوق البلد_textready_analysis.xlsx rows=79
  sample file [2] المنطقة الغ

,Region_Name,Region_Folder,Files_Read_OK,Files_Read_Err,Files_with_Text,Files_with_Stars,Files_with_Date,Total_Rows_This_Region
0,المنطقة الجنوبية,المنطقة الجنوبية,102,0,102,102,102,2701
1,المنطقة الوسطى,المنطقة الوسطى,80,0,80,80,80,1366
2,المنطقة الغربية,المنطقة الغربية,26,0,26,26,26,1358
3,المنطقة الشمالية,المنطقة الشمالية,77,0,77,77,77,1274
4,المنطقة الشرقية,المنطقة الشرقية,11,0,11,11,11,581



[STEP 1 RESULT] df1 shape: (7280, 8)
Columns: ['Text_TR', 'Stars', 'Date', 'Region_Folder', 'Region_Name', 'City_Folder', 'Source_File', '__path__']


In [7]:
df1['Stars'].value_counts()

Stars
5    3871
1    2159
3    1250
Name: count, dtype: int64

In [8]:
df1

,Text_TR,Stars,Date,Region_Folder,Region_Name,City_Folder,Source_File,__path__
0,زرتها جميله جدا وفيها قصر ابوشاهره,5,2024-06-20T12:56:45.000Z,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
1,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,1,2025-02-26T08:17:33.000Z,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
2,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,1,2024-06-19T21:47:13.000Z,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
3,كوفي ملعقه حلو يستاهل حتي لو مااعجبتك القريه,5,2024-06-19T23:53:24.000Z,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
4,انا رحت المكان مرره حلووه وفيه ٤ كوفيهات تقريب...,5,2024-08-03T13:47:02.000Z,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
...,...,...,...,...,...,...,...,...
7275,ما ادفع اكثرمن ريالين بس،، سلامات اللي جوا الم...,1,2023-11-07T06:47:17.000Z,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_tiktok-comments-scraper_2025-11-05_15-...,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
7276,حرام حتي مافيه كائنات حيه كلها قرابيع,1,2023-11-07T03:58:13.000Z,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_tiktok-comments-scraper_2025-11-05_15-...,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
7277,غواصه اشوف قاع البحر غواصه اشوف الاصنام,1,2023-11-07T13:02:11.000Z,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_tiktok-comments-scraper_2025-11-05_15-...,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...
7278,غواصه في بحر كذبي و كائنات كذبيه,1,2023-11-07T12:48:44.000Z,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_tiktok-comments-scraper_2025-11-05_15-...,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...


In [9]:
# =========================================================
# (2) FILTER ONLY TRULY-EMPTY Text_TR (NaN + "" + "nan" + "[]" ...)
# =========================================================
print(f"\n[STEP 2] Using {TEXT_COL} + strong empty filter")

# ✅ الداتا الجديدة
if TEXT_COL not in df1.columns:
    raise ValueError(f"عمود {TEXT_COL} غير موجود في البيانات.")

before2 = len(df1)

# لا نحول إلى string الآن (مهم)
s = df1[TEXT_COL]

empty_like = {
    "", "nan", "NaN", "none", "None", "NONE",
    "<NA>", "[]", "[ ]", "{}", "null", "NULL"
}

empty_mask = (
    s.isna() |
    s.astype(str).str.strip().isin(empty_like)
)

empty_count = int(empty_mask.sum())

# ✅ الناتج الجديد
df2 = df1.loc[~empty_mask].copy()

# العمود الذي سيستخدم لاحقاً للمودل
df2["TEXT_FOR_MODEL"] = df2[TEXT_COL].astype(str).str.strip()

print("\n[STEP 2 RESULT]")
print("Rows before:", before2)
print("Empty-like rows:", empty_count)
print("Rows after:", len(df2))

print("\nSample empty-like values:")
print(s.loc[empty_mask].head(10).astype(str).tolist())


[STEP 2] Using Text_TR + strong empty filter

[STEP 2 RESULT]
Rows before: 7280
Empty-like rows: 0
Rows after: 7280

Sample empty-like values:
[]


In [10]:
# =========================================================
# (4) SAFE TEXT CLEANING (INDEPENDENT)
#   - لا يعتمد على Stars_num أو y_bin
#   - يحافظ على الأعمدة التعريفية
# =========================================================

import re
import pandas as pd

print("\n[STEP 4] Safe Cleaning (Independent)")

TEXT_COL = "Text_TR"

if TEXT_COL not in df2.columns:
    raise ValueError(f"{TEXT_COL} غير موجود في df2")

# =========================
# Regex definitions
# =========================
AR_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670]")
AR_TATWEEL_RE    = re.compile(r"\u0640")

# =========================
# Safe Arabic normalization
# =========================
def normalize_arabic_safe(text: str) -> str:
    text = AR_DIACRITICS_RE.sub("", text)   # remove diacritics
    text = AR_TATWEEL_RE.sub("", text)      # remove tatweel (ـ)
    text = re.sub(r"[إأآا]", "ا", text)      # unify alef only
    return text

# =========================
# Main cleaning function
# =========================
def clean_text(s):
    if pd.isna(s):
        return ""
    
    s = str(s).strip().lower()
    if not s:
        return ""

    # remove links / emails / mentions / hashtags
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"\S+@\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"#\w+", " ", s)

    # remove emojis
    s = re.sub(r"[\U00010000-\U0010ffff]", " ", s)

    # keep Arabic / English / digits / spaces
    s = re.sub(r"[^0-9a-z\u0600-\u06FF\s]", " ", s)

    # safe normalize
    s = normalize_arabic_safe(s)

    # reduce repeated letters (جمييييل -> جمييل)
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)

    # collapse spaces
    s = re.sub(r"\s+", " ", s).strip()

    # remove if numbers only
    if re.fullmatch(r"\d+", s):
        return ""

    return s


# =========================
# Preserve metadata columns if exist
# =========================
META_COLS = ["Stars","Region_Name", "Region_Folder", "City_Folder", "Source_File", "__path__","CategoryName","Lat","Lng","Date"]
meta_existing = [c for c in META_COLS if c in df2.columns]

# =========================
# BEFORE dataframe
# =========================
df_before_clean = df2[[TEXT_COL] + meta_existing].copy()
df_before_clean = df_before_clean.rename(columns={TEXT_COL: "text_before"})

display(df_before_clean.head(10))


# =========================
# Apply cleaning
# =========================
df4 = df2.copy()
df4["text_clean"] = df4[TEXT_COL].apply(clean_text)

# =========================
# Compare dataframe
# =========================
df_compare = df4[[TEXT_COL, "text_clean"] + meta_existing].copy()
df_compare = df_compare.rename(columns={TEXT_COL: "text_before"})

display(df_compare.head(20))


# =========================
# Removed rows (empty after clean)
# =========================
df_removed = df_compare[df_compare["text_clean"].str.len() == 0].copy()

print("Removed rows (empty after clean):", len(df_removed))
display(df_removed.head(20))


# =========================
# Final cleaned dataframe
# =========================
df4 = df_compare[df_compare["text_clean"].str.len() > 0].copy()

print("Final cleaned shape:", df4.shape)
print("Columns:", df4.columns.tolist())

display(df4.head(20))


# =========================
# Safety check examples
# =========================
print("\n[CHECK] safe normalization examples:")
for t in ["سيئ", "سيئة", "سيء", "المكان سيئ جدا", "المكان رائع جدا", "12345", "مطعم 123"]:
    print(f"{t}  ->  {clean_text(t)}")


[STEP 4] Safe Cleaning (Independent)


,text_before,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,Date
0,زرتها جميله جدا وفيها قصر ابوشاهره,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-20T12:56:45.000Z
1,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2025-02-26T08:17:33.000Z
2,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T21:47:13.000Z
3,كوفي ملعقه حلو يستاهل حتي لو مااعجبتك القريه,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T23:53:24.000Z
4,انا رحت المكان مرره حلووه وفيه ٤ كوفيهات تقريب...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-08-03T13:47:02.000Z
5,منطقة السحاب جميلة اللي يروحها لابد من بدري يع...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-07-18T08:31:07.000Z
6,عادي المكان جيته وجنبهم مطرب مدري كيفبس والله ...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-28T11:19:26.000Z
7,جميل مافيه خسارة الا الكوفيهات,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-07-07T01:17:31.000Z
8,جيت انا بس خدمات غاليه ومافيها شي جميل بس جيت ...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-23T13:28:26.000Z
9,شوف الناس الذوق مو لاتجون حر,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T22:13:13.000Z


,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,Date
0,زرتها جميله جدا وفيها قصر ابوشاهره,زرتها جميله جدا وفيها قصر ابوشاهره,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-20T12:56:45.000Z
1,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2025-02-26T08:17:33.000Z
2,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T21:47:13.000Z
3,كوفي ملعقه حلو يستاهل حتي لو مااعجبتك القريه,كوفي ملعقه حلو يستاهل حتي لو مااعجبتك القريه,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T23:53:24.000Z
4,انا رحت المكان مرره حلووه وفيه ٤ كوفيهات تقريب...,انا رحت المكان مرره حلووه وفيه ٤ كوفيهات تقريب...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-08-03T13:47:02.000Z
5,منطقة السحاب جميلة اللي يروحها لابد من بدري يع...,منطقة السحاب جميلة اللي يروحها لابد من بدري يع...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-07-18T08:31:07.000Z
6,عادي المكان جيته وجنبهم مطرب مدري كيفبس والله ...,عادي المكان جيته وجنبهم مطرب مدري كيفبس والله ...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-28T11:19:26.000Z
7,جميل مافيه خسارة الا الكوفيهات,جميل مافيه خسارة الا الكوفيهات,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-07-07T01:17:31.000Z
8,جيت انا بس خدمات غاليه ومافيها شي جميل بس جيت ...,جيت انا بس خدمات غاليه ومافيها شي جميل بس جيت ...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-23T13:28:26.000Z
9,شوف الناس الذوق مو لاتجون حر,شوف الناس الذوق مو لاتجون حر,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T22:13:13.000Z


Removed rows (empty after clean): 0


,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,Date


Final cleaned shape: (7280, 9)
Columns: ['text_before', 'text_clean', 'Stars', 'Region_Name', 'Region_Folder', 'City_Folder', 'Source_File', '__path__', 'Date']


,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,Date
0,زرتها جميله جدا وفيها قصر ابوشاهره,زرتها جميله جدا وفيها قصر ابوشاهره,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-20T12:56:45.000Z
1,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2025-02-26T08:17:33.000Z
2,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T21:47:13.000Z
3,كوفي ملعقه حلو يستاهل حتي لو مااعجبتك القريه,كوفي ملعقه حلو يستاهل حتي لو مااعجبتك القريه,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T23:53:24.000Z
4,انا رحت المكان مرره حلووه وفيه ٤ كوفيهات تقريب...,انا رحت المكان مرره حلووه وفيه ٤ كوفيهات تقريب...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-08-03T13:47:02.000Z
5,منطقة السحاب جميلة اللي يروحها لابد من بدري يع...,منطقة السحاب جميلة اللي يروحها لابد من بدري يع...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-07-18T08:31:07.000Z
6,عادي المكان جيته وجنبهم مطرب مدري كيفبس والله ...,عادي المكان جيته وجنبهم مطرب مدري كيفبس والله ...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-28T11:19:26.000Z
7,جميل مافيه خسارة الا الكوفيهات,جميل مافيه خسارة الا الكوفيهات,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-07-07T01:17:31.000Z
8,جيت انا بس خدمات غاليه ومافيها شي جميل بس جيت ...,جيت انا بس خدمات غاليه ومافيها شي جميل بس جيت ...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-23T13:28:26.000Z
9,شوف الناس الذوق مو لاتجون حر,شوف الناس الذوق مو لاتجون حر,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T22:13:13.000Z



[CHECK] safe normalization examples:
سيئ  ->  سيئ
سيئة  ->  سيئة
سيء  ->  سيء
المكان سيئ جدا  ->  المكان سيئ جدا
المكان رائع جدا  ->  المكان رائع جدا
12345  ->  
مطعم 123  ->  مطعم 123


In [11]:
import re

# =========================
# قاموس الجوانب (Arabic-only) - موسع
# =========================
ASPECT_PATTERNS = {

    # 1) النظافة
    "النظافة": [
        # كلمات مباشرة
        r"(نظافه|نظافه\s*عامه|نظافه\s*المكان|نظافه\s*المحل|نظافه\s*المطعم|نظافه\s*الفرع)",
        r"(نظيف|نظيفه|نظيفين|نظيفون|نظيفات|نظيفه\s*جدا|نظيف\s*جدا|نظيف\s*مره|نظيف\s*للغا[يى]ه)",
        r"(متسخ|متسخه|متسخين|متسخات|غير\s*نظيف|مو\s*نظيف|مش\s*نظيف|ما\s*هو\s*نظيف|ماهي\s*نظيفه)",
        # قذارة/وسخ/وصخ/زبالة
        r"(وسخ|وسخه|وسخان|وسخين|وسخات|وصخ|وصخه|وصخان|وصخين|قذر|قذره|قذرين|قذارات|قذاره)",
        r"(زباله|زباله\s*بالارض|قمامه|قمامه\s*موجوده|نفايات|مخلفات)",
        # تعقيم/روائح مرتبطة بالنظافة
        r"(تعقيم|معقم|تطهير|مطهر|منظف|كلور)",
        r"(ريحه\s*وصخه|ريحه\s*كريهه|ريحه\s*مو\s*زينه|روائح\s*كريهه|زفاره|نتن|معفن)",
        # أرضيات/طاولات/مقاعد
        r"(ارضيه|ارضيات|الارض|الارضيه|طاولات\s*وصخه|الطاولات\s*وصخه|كراسي\s*وصخه|المقاعد\s*وصخه)",
        # حشرات كجزء من النظافة (بدون إنشاء جانب مستقل)
        r"(حشره|حشرات|نمل|ذباب|صراصير|صرصور)"
    ],

    # 2) دورات المياه (مفصول عن النظافة لأنه جانب مستقل لديك)
    "دورات المياه": [
        r"(دورات\s*المياه|دوره\s*مياه|دوره\s*المي[اه]ه|دورات\s*المي[اه]ه)",
        r"(حمام|الحمام|حمامات|دورات|تو[اا]ليت|مرحاض|مراحيض)",
        # توفر/صلاحية
        r"(ما\s*في\s*حمام|مافي\s*حمام|ما\s*في\s*دورات|مافي\s*دورات|الحمام\s*مقف[لو]ل|الحمام\s*مقفل)",
        r"(الحمام\s*خربان|المرحاض\s*خربان|السيفون\s*خربان|مغسله\s*خربانه|المغسله\s*خربانه)",
        r"(مويه|ماء|مياه|حنفيه|صنبور|مغسله|مغاسل|مغسله\s*يدين|مكان\s*غسيل)",
        r"(صابون|معقم|مناديل|محارم|ورق\s*تواليت|منشفه|مجفف)",
        # روائح/اتساخ داخل الحمام
        r"(حمام\s*وسخ|حمام\s*وصخ|حمام\s*قذر|حمام\s*متسخ|دورات\s*وسخه|دورات\s*وصخه|الحمامات\s*وسخه)",
        r"(ريحة\s*الحمام|زفاره\s*الحمام|روائح\s*الحمام|الحمام\s*ريحته\s*كريهه)",
        # ازدحام/حجم
        r"(زحمه\s*الحمام|طابور\s*الحمام|انتظار\s*الحمام|حمام\s*ضيق|حمامات\s*قليله)"
    ],

    # 3) الخدمة
    "الخدمة": [
        r"(خدمه|الخدمه|مستوى\s*الخدمه|جوده\s*الخدمه|خدمه\s*العملاء)",
        r"(تعامل|التعامل|اسلوب|الاسلوب|معامله|المعامله|طريقه\s*التعامل|التجاوب|الاهتمام)",
        r"(استقبال|الاستقبال|الاستقب[اآ]ل|ترحيب|الترحيب)",
        r"(موظف|موظفين|العامل|العمال|الطاقم|الكاشير|الكاشيره|الصراف|المحاسب)",
        r"(مدير|المدير|الاداره|الاداره|المشرف|مسؤول)",
        # صفات إيجابية/سلبية مرتبطة بالخدمة
        r"(متعاون|متعاونين|تعاون|لبق|محترم|احترام|را[يى]ق|بشوش|مبتسم)",
        r"(وقح|وقاحه|قله\s*ادب|عدم\s*احترام|سيء\s*التعامل|سوء\s*التعامل|اسلوب\s*سيء|تعامل\s*سيء|تجاهل)",
        r"(ما\s*يرد|ما\s*يردون|ما\s*يردون\s*علينا|يردون\s*ببطء|رد\s*متاخر|رد\s*بطيء)",
        # أخطاء خدمة
        r"(خدمه\s*سيئه|خدمه\s*ضعيفه|خدمه\s*ممتازه|خدمه\s*رائعه|خدمه\s*سريعه|خدمه\s*بطيئه)"
    ],

    # 4) مواقف السيارات
    "مواقف السيارات": [
        r"(مواقف|موقف|مواقف\s*السيارات|موقف\s*السياره|موقف\s*السيارات)",
        r"(باركنج|مواقف\s*قليله|مواقف\s*محدوده|ما\s*في\s*مواقف|مافي\s*مواقف|بدون\s*مواقف)",
        r"(صعب\s*تلقى\s*موقف|صعب\s*احصل\s*موقف|تدوير\s*على\s*موقف|لفات\s*على\s*موقف)",
        r"(زحمه\s*مواقف|ازدحام\s*المواقف|المواقف\s*زحمه|المواقف\s*مكتظه)",
        r"(مواقف\s*بعيده|مواقف\s*قريبه|مواقف\s*مظلمه|مواقف\s*غير\s*مريحه)",
        r"(مواقف\s*ترابيه|مواقف\s*غير\s*مرتبه|مواقف\s*غير\s*منظمه|تنظيم\s*المواقف)",
        r"(مواقف\s*مخصصه|مواقف\s*خاصه|مواقف\s*للعائلات|مواقف\s*لذوي\s*الاحتياجات)"
    ],

    # 5) الانتظار
    "الانتظار": [
        r"(انتظار|الانتظار|وقت\s*انتظار|مدة\s*انتظار|طابور|صف|الدور)",
        r"(تاخير|تأخير|تت[اأ]خر|يت[اأ]خر|متاخر|متأخر|تأخر)",
        r"(ياخذ\s*وقت|ياخذ\s*وقته|طول\s*وقت|وقت\s*طويل|طولنا|تطويل)",
        r"(بطيء|بطيئ|بط[يى]ء|بطئ|بطيئه|بطيئه\s*جدا|بط[يى]ء\s*جدا)",
        r"(سرعه|سريع|سريعه|سريعين|فوري|بدون\s*انتظار)",
        r"(تجهيز|تجهيز\s*بطيء|تجهيز\s*سريع|تاخير\s*التجهيز)"
    ],

    # 6) الأسعار
    "الأسعار": [
        r"(سعر|اسعار|الاسعار|تسعيره|التسعيره|قيمة|القيمه|قيمة\s*مقابل|قيمة\s*السعر)",
        r"(غالي|غاليه|غاليين|غاليه\s*جدا|غالي\s*جدا|مرتف[اعه]|مرتفع)",
        r"(مبالغ\s*فيه|مبالغ\s*فيها|مبالغه|استغلال|ينه[بب]ون|غلاء)",
        r"(رخيص|رخيصه|رخيصين|رخيصه\s*جدا|رخيص\s*جدا|مناسب|مناسبه|اسعار\s*مناسبه)",
        r"(يستاهل|ما\s*يستاهل|ما\s*يسوى|لا\s*يسوى|قيمه\s*ممتازه|قيمه\s*جيده)",
        r"(عروض|خصم|تخفيض|تخفيضات|اسعار\s*العروض)"
    ],

    # 7) الطعام
    "الطعام": [
        r"(اكل|الاكل|طعام|الطعام|وجبه|وجبات|صحن|اطباق|طبق)",
        r"(طعم|طعمه|مذاق|نكهه|نكهة)",
        r"(لذيذ|لذيذه|لذيذ\s*جدا|لذيذ\s*مره|شهي|يشهي|ممتاز\s*الطعم)",
        r"(جوده|الجوده|جودة|مستوى|مستوى\s*الاكل|مستوى\s*الطعام)",
        r"(طازج|مو\s*طازج|غير\s*طازج|فريش|قديم|بايت)",
        r"(بارد|حار\s*مره|محروق|يابس|ناشف|مستوي\s*زيادة|مستوي\s*ناقص)",
        r"(دهني|زيت|زيت\s*كثير|مالح|ملح\s*زايد|حار\s*زياده|سبايسي\s*زياده)",
        r"(كميه|كمية|حجم|حجم\s*صغير|حجم\s*كبير|قليل|كثير)",
        r"(تتبيل|بهارات|بهار|صلصه|صوص)",
        r"(غير\s*لذيذ|مو\s*لذيذ|طعم\s*سيء|طعمه\s*سيء|ماله\s*طعم)"
    ],

    # 8) الإضاءة
    "الإضاءة": [
        r"(اضاءه|اضاءه\s*المكان|اضاءه\s*ضعيفه|اضاءه\s*قويه|اضاءه\s*خفيفه|اضاءه\s*مزعجه)",
        r"(إضاءه|إضاءه\s*ضعيفه|إضاءه\s*قويه|إضاءه\s*مزعجه)",  # لو ما وحّدت الألف/الهمزات
        r"(اضاءه\s*سيئه|اضاءه\s*حلوه|اضاءه\s*جميله|اضاءه\s*ممتازه)",
        r"(نور|الانوار|اناره|اناره\s*ضعيفه|اناره\s*قويه)",
        r"(لمبه|لمبات|اللمبات|سبوت|كشاف|كشافات)",
        r"(مظلم|ظلام|معتم|الاضاءه\s*مطف[يى]ه|الاضاءه\s*ضعيفه\s*جدا)",
        r"(انعكاس|يبهر|بهار|تعم[يى]|يوجع\s*العين)"
    ],

    # 9) الزحمة
    "الزحمة": [
        r"(زحمه|زحام|ازدحام|مزدحم|مكتظ|كتمه|خانقه|خانقه\s*مره)",
        r"(مكان\s*زحمه|الفرع\s*زحمه|المحل\s*زحمه|المطعم\s*زحمه)",
        r"(ازدحام\s*شديد|زحمه\s*شديده|زحمه\s*قويه)",
        r"(طاولات\s*قريبه|قريبين\s*من\s*بعض|مساحه\s*ضيقه|ضيق)",
        r"(ما\s*في\s*جلسات|جلسات\s*قليله|اماكن\s*قليله)",
        r"(صعب\s*تلقى\s*مكان|ما\s*لقيت\s*مكان|مو\s*حاصل\s*مكان)"
    ],
    
    "الألعاب": [
    r"(العاب|ألعاب|منطقه\s*العاب|منطقة\s*ألعاب|قسم\s*العاب|صاله\s*العاب|صالة\s*ألعاب)",
    r"(ملاهي|ملاه[يى]|ترفيه|ترفيه[يى]|أركيد|اركيد)",
    r"(بلايستيشن|بلاي\s*ستيشن|سوني|اكس\s*بوكس|نينتندو)",
    r"(طاول[هة]\s*بلياردو|بلياردو|بولينج|بولنج|هوكي|اير\s*هوكي)",
    r"(العاب\s*اطفال|ألعاب\s*أطفال|منطقه\s*اطفال|منطقة\s*أطفال|العاب\s*صغار)",
    r"(زحمه\s*العاب|ازدحام\s*العاب|انتظار\s*العاب|طابور\s*العاب)",
    r"(اجهزه\s*العاب|أجهزة\s*ألعاب|مكائن\s*العاب|مكائن\s*ألعاب|ألعاب\s*الكترونيه)",
    r"(تذاكر\s*العاب|كروت\s*العاب|بطاق[هة]\s*العاب|شحن\s*الكرت)"
],

"الصيانة": [
    r"(صيان[هة]|صيانه|صيانه\s*المكان|صيانه\s*المحل|الصيانه|فني|فنيين)",
    r"(عطل|اعطال|أعطال|خربان|خربانه|خربانه\s*مره|مخرب|مو\s*شغال|مش\s*شغال|لا\s*يعمل|ما\s*يشتغل)",
    r"(تصليح|اصلاح|إصلاح|تصليحات|إصلاحات|تبديل|تغيير|استبدال)",
    r"(سباك[هة]|سباك|كهرب[اءا]ء|كهربائي|كهربائيه|تمديدات|تسريب|تهريب)",
    r"(مكيف\s*خربان|التكييف\s*خربان|لمبات\s*خربانه|الاضاءه\s*خربانه|مصعد\s*خربان|دوره\s*مياه\s*خربانه)",
    r"(صيان[هة]\s*ضعيفه|ما\s*في\s*صيان[هة]|تاخير\s*الصيان[هة]|تأخير\s*الصيان[هة])"
],
}

# =========================
# Compile regex (important for speed)
# =========================
ASPECT_REGEX = {
    asp: [re.compile(pat) for pat in pats]
    for asp, pats in ASPECT_PATTERNS.items()
}

In [12]:
def detect_aspects_with_hits(text: str):

    if not isinstance(text, str):
        return {}

    found = {}

    for asp, regs in ASPECT_REGEX.items():
        hits = []

        for rg in regs:
            for m in rg.finditer(text):
                hits.append(m.group(0))

        if hits:
            found[asp] = sorted(set(hits))

    return found

df4["aspect_hits"] = df4["text_clean"].apply(detect_aspects_with_hits)

In [13]:
df4

,text_before,text_clean,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,Date,aspect_hits
0,زرتها جميله جدا وفيها قصر ابوشاهره,زرتها جميله جدا وفيها قصر ابوشاهره,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-20T12:56:45.000Z,{}
1,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2025-02-26T08:17:33.000Z,"{'الطعام': ['كثير'], 'الألعاب': ['العاب']}"
2,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T21:47:13.000Z,"{'الخدمة': ['رايق'], 'الزحمة': ['ضيق']}"
3,كوفي ملعقه حلو يستاهل حتي لو مااعجبتك القريه,كوفي ملعقه حلو يستاهل حتي لو مااعجبتك القريه,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T23:53:24.000Z,{'الأسعار': ['يستاهل']}
4,انا رحت المكان مرره حلووه وفيه ٤ كوفيهات تقريب...,انا رحت المكان مرره حلووه وفيه ٤ كوفيهات تقريب...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-08-03T13:47:02.000Z,{}
...,...,...,...,...,...,...,...,...,...,...
7275,ما ادفع اكثرمن ريالين بس،، سلامات اللي جوا الم...,ما ادفع اكثرمن ريالين بس،، سلامات اللي جوا الم...,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_tiktok-comments-scraper_2025-11-05_15-...,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2023-11-07T06:47:17.000Z,"{'دورات المياه': ['مويه'], 'الأسعار': ['يستاهل']}"
7276,حرام حتي مافيه كائنات حيه كلها قرابيع,حرام حتي مافيه كائنات حيه كلها قرابيع,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_tiktok-comments-scraper_2025-11-05_15-...,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2023-11-07T03:58:13.000Z,{}
7277,غواصه اشوف قاع البحر غواصه اشوف الاصنام,غواصه اشوف قاع البحر غواصه اشوف الاصنام,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_tiktok-comments-scraper_2025-11-05_15-...,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2023-11-07T13:02:11.000Z,{}
7278,غواصه في بحر كذبي و كائنات كذبيه,غواصه في بحر كذبي و كائنات كذبيه,1,المنطقة الوسطى,المنطقة الوسطى,الرياض,dataset_tiktok-comments-scraper_2025-11-05_15-...,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2023-11-07T12:48:44.000Z,{}


In [14]:
# ======================================================
# CREATE ASPECT LONG DATAFRAME (ONE ASPECT PER ROW)
# ======================================================

TEXT_COL = "text_clean"   # أو Text_TR

META_COLS = ["Stars","Region_Name", "Region_Folder", "City_Folder", "Source_File", "__path__","Date"]
meta_existing = [c for c in META_COLS if c in df4.columns]

df4 = df4.copy()

# استخراج الجوانب
df4["aspect_hits"] = df4[TEXT_COL].apply(detect_aspects_with_hits)

# تحويل إلى صف لكل جانب
rows = []

for idx, r in df4.iterrows():

    hits_dict = r["aspect_hits"]

    if not hits_dict:
        continue

    for aspect, hits in hits_dict.items():

        rows.append({
            "row_id": idx,
            "aspect": aspect,
            "hits": " | ".join(hits),
            "text": r[TEXT_COL],
            **{c: r[c] for c in meta_existing}
        })

df_aspect_long = pd.DataFrame(rows)

print("Aspect-level rows:", len(df_aspect_long))
display(df_aspect_long.head(20))

print("\nAspect distribution:")
display(df_aspect_long["aspect"].value_counts())

Aspect-level rows: 3597


,row_id,aspect,hits,text,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,Date
0,1,الطعام,كثير,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2025-02-26T08:17:33.000Z
1,1,الألعاب,العاب,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2025-02-26T08:17:33.000Z
2,2,الخدمة,رايق,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T21:47:13.000Z
3,2,الزحمة,ضيق,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T21:47:13.000Z
4,3,الأسعار,يستاهل,كوفي ملعقه حلو يستاهل حتي لو مااعجبتك القريه,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T23:53:24.000Z
5,5,الطعام,اكل,منطقة السحاب جميلة اللي يروحها لابد من بدري يع...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-07-18T08:31:07.000Z
6,8,الأسعار,غالي,جيت انا بس خدمات غاليه ومافيها شي جميل بس جيت ...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-23T13:28:26.000Z
7,10,الطعام,بارد,والله البارح رحنا السوده مقفله وعشان الجو البا...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-28T08:16:47.000Z
8,11,الأسعار,رخيص,عندنا شاليهات رخيصه وحلوه,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2025-07-16T10:48:35.000Z
9,12,النظافة,غير نظيف | نظيف,جناها مافيها شي يذكر مجموعة غرف مسكره وغير مرم...,3,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-30T22:09:41.000Z



Aspect distribution:


aspect
الأسعار           1296
الطعام             831
النظافة            384
الخدمة             270
الزحمة             180
دورات المياه       161
الانتظار           128
الألعاب            126
الإضاءة            112
الصيانة             73
مواقف السيارات      36
Name: count, dtype: int64

In [15]:
aspect_overall = (
    df_aspect_long
    .groupby("aspect")
    .size()
    .reset_index(name="mentions")
    .sort_values("mentions", ascending=False)
)

display(aspect_overall)

,aspect,mentions
0,الأسعار,1296
7,الطعام,831
8,النظافة,384
4,الخدمة,270
5,الزحمة,180
9,دورات المياه,161
3,الانتظار,128
1,الألعاب,126
2,الإضاءة,112
6,الصيانة,73


In [16]:
total_mentions = aspect_overall["mentions"].sum()

aspect_overall["percentage_%"] = (
    aspect_overall["mentions"] / total_mentions * 100
).round(2)

display(aspect_overall)

,aspect,mentions,percentage_%
0,الأسعار,1296,36.03
7,الطعام,831,23.10
8,النظافة,384,10.68
4,الخدمة,270,7.51
5,الزحمة,180,5.00
9,دورات المياه,161,4.48
3,الانتظار,128,3.56
1,الألعاب,126,3.50
2,الإضاءة,112,3.11
6,الصيانة,73,2.03


In [17]:
aspect_unique_reviews = (
    df_aspect_long
    .groupby("aspect")["row_id"]
    .nunique()
    .reset_index(name="unique_reviews")
    .sort_values("unique_reviews", ascending=False)
)

display(aspect_unique_reviews)

,aspect,unique_reviews
0,الأسعار,1296
7,الطعام,831
8,النظافة,384
4,الخدمة,270
5,الزحمة,180
9,دورات المياه,161
3,الانتظار,128
1,الألعاب,126
2,الإضاءة,112
6,الصيانة,73


In [18]:
aspect_by_region = (
    df_aspect_long
    .groupby(["Region_Name", "aspect"])
    .size()
    .reset_index(name="mentions")
    .sort_values(["Region_Name","mentions"], ascending=[True, False])
)

display(aspect_by_region.head(30))

,Region_Name,aspect,mentions
0,المنطقة الجنوبية,الأسعار,429
7,المنطقة الجنوبية,الطعام,339
8,المنطقة الجنوبية,النظافة,144
4,المنطقة الجنوبية,الخدمة,94
5,المنطقة الجنوبية,الزحمة,67
3,المنطقة الجنوبية,الانتظار,52
9,المنطقة الجنوبية,دورات المياه,46
1,المنطقة الجنوبية,الألعاب,44
2,المنطقة الجنوبية,الإضاءة,35
6,المنطقة الجنوبية,الصيانة,25


In [19]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [20]:
# =========================
# CONFIG
# =========================
MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"  # مثال شائع
MAX_LEN = 128
BATCH_SIZE = 64   # جرّب 64 على GPU، إذا حصل OOM خفّض لـ 32
SAT_MODE = "pos_only"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# (2) DEVICE + LOAD MODEL
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print("id2label:", model.config.id2label)

# Identify label indices robustly
id2label = {int(k): v for k, v in model.config.id2label.items()}
label_lower = {i: str(lab).lower() for i, lab in id2label.items()}

def _find_idx(keys):
    for i, lab in label_lower.items():
        if any(k in lab for k in keys):
            return i
    return None

pos_idx = _find_idx(["pos", "positive", "ايجاب", "إيجاب"])
neu_idx = _find_idx(["neu", "neutral", "محايد"])
neg_idx = _find_idx(["neg", "negative", "سلب", "سلبي"])

# Fallback if model uses common ordering but labels are generic like LABEL_0/1/2
if pos_idx is None or neu_idx is None or neg_idx is None:
    # Most common ordering in 3-class sentiment: 0=negative, 1=neutral, 2=positive
    # If your model differs, print(model.config.id2label) and adjust here.
    pos_idx, neu_idx, neg_idx = 2, 1, 0

print("Indices -> pos:", pos_idx, "neu:", neu_idx, "neg:", neg_idx)

Device: cpu
Device: cpu


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


id2label: {0: 'positive', 1: 'negative', 2: 'neutral'}
Indices -> pos: 0 neu: 2 neg: 1


In [21]:
# =========================
# (3) BUILD BERT INPUT (Aspect-aware)
# =========================
# Each row: aspect [SEP] text
df_aspect_long["bert_input"] = (
    df_aspect_long["aspect"].astype(str) + " [SEP] " + df_aspect_long["text"].astype(str)
)

# Optional: drop empty
df_aspect_long = df_aspect_long[df_aspect_long["bert_input"].str.len() > 0].copy()
df_aspect_long.reset_index(drop=True, inplace=True)

print("After bert_input:", df_aspect_long.shape)


After bert_input: (3597, 12)


In [22]:
# =========================
# (4) BATCH INFERENCE FUNCTION
# =========================
def batch_predict_3class(texts, batch_size=64, max_len=128, log_every_batches=300):
    """
    Returns:
      p_pos, p_neu, p_neg (np arrays length N)
    """
    all_pos, all_neu, all_neg = [], [], []

    N = len(texts)
    for b, start in enumerate(range(0, N, batch_size), 1):
        batch_texts = texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            logits = model(**enc).logits  # [B, 3]

        probs = F.softmax(logits, dim=1)  # [B, 3]
        probs = probs.detach().cpu().numpy()

        all_pos.append(probs[:, pos_idx])
        all_neu.append(probs[:, neu_idx])
        all_neg.append(probs[:, neg_idx])

        if (b % log_every_batches) == 0:
            done = min(start + batch_size, N)
            print(f"Processed {done}/{N} rows...")

    p_pos = np.concatenate(all_pos, axis=0)
    p_neu = np.concatenate(all_neu, axis=0)
    p_neg = np.concatenate(all_neg, axis=0)
    return p_pos, p_neu, p_neg

# =========================
# (5) RUN INFERENCE ON 300k+
# =========================
texts = df_aspect_long["bert_input"].tolist()

p_pos, p_neu, p_neg = batch_predict_3class(
    texts,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
    log_every_batches=300
)

df_aspect_long["p_positive"] = p_pos
df_aspect_long["p_neutral"]  = p_neu
df_aspect_long["p_negative"] = p_neg

# Final label by argmax
pred_idx = np.argmax(np.stack([df_aspect_long["p_negative"],
                               df_aspect_long["p_neutral"],
                               df_aspect_long["p_positive"]], axis=1), axis=1)
# The stack above is [neg, neu, pos] => idx 0/1/2
df_aspect_long["sentiment"] = np.where(pred_idx == 2, "ايجابي",
                               np.where(pred_idx == 1, "محايد", "سلبي"))

# Satisfaction score
if SAT_MODE == "pos_only":
    df_aspect_long["satisfaction_score"] = df_aspect_long["p_positive"]
elif SAT_MODE == "pos_plus_half_neu":
    df_aspect_long["satisfaction_score"] = df_aspect_long["p_positive"] + 0.5 * df_aspect_long["p_neutral"]
else:
    raise ValueError("SAT_MODE غير صحيح. استخدم: pos_only أو pos_plus_half_neu")

print("Inference done.")
display(df_aspect_long.head(10))

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Inference done.


,row_id,aspect,hits,text,Stars,Region_Name,Region_Folder,City_Folder,Source_File,__path__,Date,bert_input,p_positive,p_neutral,p_negative,sentiment,satisfaction_score
0,1,الطعام,كثير,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2025-02-26T08:17:33.000Z,الطعام [SEP] والله الجو يجنن بس الافكار مافي ز...,0.121026,0.111127,0.767847,سلبي,0.121026
1,1,الألعاب,العاب,والله الجو يجنن بس الافكار مافي زحمة كافيهات و...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2025-02-26T08:17:33.000Z,الألعاب [SEP] والله الجو يجنن بس الافكار مافي ...,0.127667,0.106797,0.765536,سلبي,0.127667
2,2,الخدمة,رايق,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T21:47:13.000Z,الخدمة [SEP] مجانا و طبعا دامها عندنا اكثر من ...,0.197560,0.357764,0.444677,سلبي,0.197560
3,2,الزحمة,ضيق,مجانا و طبعا دامها عندنا اكثر من مرة اجيها مرة...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T21:47:13.000Z,الزحمة [SEP] مجانا و طبعا دامها عندنا اكثر من ...,0.108693,0.223006,0.668301,سلبي,0.108693
4,3,الأسعار,يستاهل,كوفي ملعقه حلو يستاهل حتي لو مااعجبتك القريه,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-19T23:53:24.000Z,الأسعار [SEP] كوفي ملعقه حلو يستاهل حتي لو ماا...,0.851716,0.137634,0.010650,ايجابي,0.851716
5,5,الطعام,اكل,منطقة السحاب جميلة اللي يروحها لابد من بدري يع...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-07-18T08:31:07.000Z,الطعام [SEP] منطقة السحاب جميلة اللي يروحها لا...,0.836743,0.159870,0.003387,ايجابي,0.836743
6,8,الأسعار,غالي,جيت انا بس خدمات غاليه ومافيها شي جميل بس جيت ...,5,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-23T13:28:26.000Z,الأسعار [SEP] جيت انا بس خدمات غاليه ومافيها ش...,0.807967,0.138701,0.053332,ايجابي,0.807967
7,10,الطعام,بارد,والله البارح رحنا السوده مقفله وعشان الجو البا...,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-28T08:16:47.000Z,الطعام [SEP] والله البارح رحنا السوده مقفله وع...,0.047829,0.069138,0.883032,سلبي,0.047829
8,11,الأسعار,رخيص,عندنا شاليهات رخيصه وحلوه,1,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2025-07-16T10:48:35.000Z,الأسعار [SEP] عندنا شاليهات رخيصه وحلوه,0.042425,0.099198,0.858377,سلبي,0.042425
9,12,النظافة,غير نظيف | نظيف,جناها مافيها شي يذكر مجموعة غرف مسكره وغير مرم...,3,المنطقة الجنوبية,المنطقة الجنوبية,بيانات منطقة عسير - بعد المعالجة,Video comments 30_textready_analysis.xlsx,/kaggle/input/datasets/ziyadaltalhi/tik-tok-da...,2024-06-30T22:09:41.000Z,النظافة [SEP] جناها مافيها شي يذكر مجموعة غرف ...,0.007391,0.020399,0.972210,سلبي,0.007391


In [23]:
# احفظ التفاصيل (كل صف = جانب داخل تعليق + احتمالات)
df_aspect_long.drop(columns=["bert_input"], errors="ignore").to_csv(
    "/kaggle/working/aspect_sentiment_detail-tiktok.csv",
    index=False,
    encoding="utf-8-sig"
)



print("Saved to /kaggle/working/")

Saved to /kaggle/working/
